# ViFinQA — merge finished shards into a submission

Nothing here needs a GPU. Merging, re-executing every query and packaging the ZIP are all
CPU work, so a run whose shards have finished can be turned into a submission in minutes,
and the only file worth downloading afterwards is the ZIP itself.

Attach three inputs:

1. `vifinqa` — the corpus, in case an evidence CSV has to be rebuilt.
2. `vifinqa-artifacts` — the frozen table manifest.
3. the output of the notebook whose shards finished.

Select **no accelerator**. Then run every cell.


In [ ]:
import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def iter_input_paths(
    relative: str, root: Path = Path("/kaggle/input"), max_depth: int = 12
) -> list[Path]:
    """Return every existing `<directory>/relative` under `root`, following symlinked mounts."""
    matches: list[Path] = []
    visited: set[str] = set()
    for parent, directories, _ in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        if len(Path(parent).parts) - len(root.parts) >= max_depth:
            directories.clear()
        candidate = Path(parent) / relative
        if candidate.exists():
            matches.append(candidate)
    return sorted(matches, key=str)


def describe_inputs(root: Path = Path("/kaggle/input"), max_depth: int = 5) -> str:
    """Return a compact inventory of mounted inputs so failures name what is actually attached.

    Kaggle spends three levels on `datasets/<owner>/<slug>` before any content, so the
    default depth has to reach past the mount point itself.
    """
    if not root.is_dir():
        return f"{root} does not exist"
    lines: list[str] = []
    visited: set[str] = set()
    for parent, directories, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        depth = len(Path(parent).parts) - len(root.parts)
        entries = sorted(filenames)[:4]
        if len(filenames) > 4:
            entries.append(f"+{len(filenames) - 4} more files")
        if depth >= max_depth:
            if directories:
                entries.append(f"+{len(directories)} more directories")
            directories.clear()
        directories.sort()
        lines.append(f"{'  ' * depth}{Path(parent).name or root}/ {entries}")
        if len(lines) >= 80:
            lines.append("... truncated")
            break
    return "\n".join(lines)


def make_writable(root: Path) -> None:
    """Kaggle mounts inputs read-only and copytree preserves that."""
    for path in [root, *root.rglob("*")]:
        path.chmod(path.stat().st_mode | (0o700 if path.is_dir() else 0o600))


INPUT_INVENTORY = describe_inputs()
print("Kaggle inputs:\n" + INPUT_INVENTORY)


In [ ]:
data_candidates = sorted(
    {
        questions.parent.parent
        for questions in iter_input_paths("questions/questions.jsonl")
        if (questions.parent.parent / "code_stock.csv").is_file()
    },
    key=str,
)
assert data_candidates, f"Attach the `vifinqa` corpus.\n{INPUT_INVENTORY}"
DATA_ROOT = data_candidates[0]
QUESTIONS = DATA_ROOT / "questions/questions.jsonl"

manifests = sorted(
    {
        manifest
        for manifest in iter_input_paths("processed/table_manifest.jsonl")
        if manifest.with_suffix(".parquet").is_file()
    },
    key=str,
)
assert manifests, f"Attach the `vifinqa-artifacts` manifest.\n{INPUT_INVENTORY}"
MANIFEST = manifests[0].with_suffix(".parquet")

SUPPORTED_PROFILES = {
    "qwen3_8b_awq": (
        "Qwen/Qwen3-8B-AWQ",
        "4da05a8edb55c6046cce958586c33b61da07bb79",
    ),
    "qwen3_14b_awq": (
        "Qwen/Qwen3-14B-AWQ",
        "31c69efc29464b6bb0aee1398b5a7b50a99340c3",
    ),
}
checkpoint_candidates = [
    (profile, path)
    for profile in SUPPORTED_PROFILES
    for path in iter_input_paths(f"generation_{profile}_shards/shard_0/run_metadata.json")
]
# Two finished sessions can be attached at once for good reason -- the earlier one may hold
# the hybrid ranking or diagnostics the later one no longer writes -- so refusing outright
# makes the user detach an input they need. Name the one you mean instead. The guard still
# refuses when nothing is named and more than one is present, because resuming the wrong run
# silently is worse than stopping.
WANTED_CHECKPOINT = os.environ.get("VIFINQA_CHECKPOINT_SOURCE", "").strip()
if WANTED_CHECKPOINT:
    checkpoint_candidates = [
        candidate for candidate in checkpoint_candidates if WANTED_CHECKPOINT in str(candidate[1])
    ]
assert len(checkpoint_candidates) == 1, (
    f"Attach exactly one supported finished checkpoint, or name one with VIFINQA_CHECKPOINT_SOURCE (a substring of its path). Found {checkpoint_candidates}.\n"
    f"{INPUT_INVENTORY}"
)
MODEL_PROFILE, prior_metadata_path = checkpoint_candidates[0]
prior_metadata = json.loads(prior_metadata_path.read_text(encoding="utf-8"))
expected_model, expected_revision = SUPPORTED_PROFILES[MODEL_PROFILE]
assert prior_metadata.get("model") == expected_model
assert prior_metadata.get("model_revision") == expected_revision


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


retrieval_candidates = [
    path
    for path in [
        *iter_input_paths("retrieval_reranked.jsonl"),
        *iter_input_paths("retrieval_hybrid.jsonl"),
    ]
    if file_sha256(path) == prior_metadata.get("retrieval_sha256")
]
assert len(retrieval_candidates) == 1, (
    "Finished checkpoint output must contain exactly its recorded retrieval."
)
RETRIEVAL = retrieval_candidates[0]
prior = [prior_metadata_path]
PROJECT_SHA = prior_metadata["project_revision"]
GENERATION_NAME = f"generation_{MODEL_PROFILE}"
GEN_SHARDS = Path("/kaggle/working/artifacts") / f"{GENERATION_NAME}_shards"
GEN = Path("/kaggle/working/artifacts") / GENERATION_NAME
if not GEN_SHARDS.exists():
    GEN_SHARDS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(prior[0].parents[1], GEN_SHARDS)
    make_writable(GEN_SHARDS)
shard_dirs = sorted(GEN_SHARDS.glob("shard_*"), key=lambda path: int(path.name.split("_")[1]))
answers = 0
for shard in shard_dirs:
    rows = [
        line
        for line in (shard / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    answers += len(rows)
print(f"{len(shard_dirs)} shards holding {answers} answers, written by {PROJECT_SHA[:12]}")
assert answers == 1012, f"Expected 1,012 answers across the shards, found {answers}."


In [ ]:
# Package with the code that produced the answers, not with whatever main points at.
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")
if PROJECT.exists() and not (PROJECT / "pyproject.toml").exists():
    shutil.rmtree(PROJECT)
if not PROJECT.exists():
    result = subprocess.run(["git", "clone", GIT_URL, str(PROJECT)], capture_output=True, text=True)
    if result.returncode:
        try:
            from kaggle_secrets import UserSecretsClient

            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as exc:
            raise RuntimeError((result.stderr or "git clone failed").strip()) from exc
        auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        git_env = os.environ.copy()
        git_env.update(
            {
                "GIT_CONFIG_COUNT": "1",
                "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
            }
        )
        subprocess.run(["git", "clone", GIT_URL, str(PROJECT)], env=git_env, check=True)
        del token, auth, git_env
subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", PROJECT_SHA], check=False)
subprocess.run(["git", "-C", str(PROJECT), "checkout", PROJECT_SHA], check=True)
# Merging and validating need pandas and pyarrow, not a serving stack.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
os.chdir(PROJECT)
print("packaging with", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
# Any evidence CSV the shards no longer carry is rebuilt from the corpus.
subprocess.run(
    [
        sys.executable,
        "scripts/52_restore_evidence_csv.py",
        *[str(path) for path in shard_dirs],
        "--manifest",
        str(MANIFEST),
        "--data-root",
        str(DATA_ROOT),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/51_merge_generation_shards.py",
        *[str(path) for path in shard_dirs],
        "--output",
        str(GEN),
        "--expected-rows",
        "1012",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/45_finalize_submission.py",
        str(GEN / "submission.json"),
        "--retrieval",
        str(RETRIEVAL),
        "--manifest",
        str(MANIFEST),
        "--output",
        str(GEN / "submission_z4_abs.json"),
    ],
    check=True,
)
FINAL_SUBMISSION = GEN / "submission_z4_abs.json"
# Re-executes every query against the evidence it cites, which is the gate that says the
# submission reproduces rather than merely parses.
subprocess.run(
    [
        sys.executable,
        "scripts/40_validate_submission.py",
        str(FINAL_SUBMISSION),
        "--questions",
        str(QUESTIONS),
        "--evidence-root",
        str(GEN),
        "--allow-partial-docs",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/41_package_submission.py",
        str(FINAL_SUBMISSION),
        "/kaggle/working/submission.zip",
        "--questions",
        str(QUESTIONS),
        "--evidence-root",
        str(GEN),
        "--allow-partial-docs",
    ],
    check=True,
)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


predictions = json.loads(FINAL_SUBMISSION.read_text(encoding="utf-8"))
traces = [
    json.loads(line)
    for line in (GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
fallbacks = [trace for trace in traces if trace.get("fallback")]
solved = len(predictions) - len(fallbacks)
submission = Path("/kaggle/working/submission.zip")
print(
    f"predictions {len(predictions)}/1012 | solved by the model {solved} "
    f"({100 * solved / len(predictions):.0f}%) | fallback {len(fallbacks)}"
)
print(submission, submission.stat().st_size, sha256(submission))
print("Download only this ZIP.")
